# Experimentos — ronda 3 · variantes de `E10_espacial_simple` · AICOLOMBIA

En la ronda 2 ganó **`E10_espacial_simple` (E10S)**: clasificador simple (100 árboles, 15 hojas, mínimo 200 filas por
hoja) + variables espaciales (población, ubicación y tasa de los vecinos). Esta ronda explora alrededor de ese enfoque:

| Grupo | Variantes | Pregunta que responde |
|---|---|---|
| Referencias | `E10S`, `E7_lento` | ¿Se reproducen los resultados de la ronda 2? |
| Ablación | `E10S_sin_poblacion`, `E10S_sin_ubicacion`, `E10S_sin_vecinos` | ¿Qué grupo de variables espaciales aporta? |
| Definición de vecinos | `E10S_k3`, `E10S_k10`, `E10S_k10_ponderado`, `E10S_frontera` | ¿Cuántos vecinos y cómo definirlos (cercanía o frontera compartida)? |
| Información de los vecinos | `E10S_vecinos_plus`, `E10S_suavizado_espacial` | ¿Sirve también su magnitud y su tasa reciente? ¿Encoger la tasa del municipio hacia sus vecinos en vez del departamento? |
| Complejidad | `E10S_arboles_50`, `E10S_arboles_150`, `E10S_hojas_7`, `E10S_lento`, `E10S_todo_simple` | ¿Dónde está el punto justo de simplicidad? |
| Varianza | `E10S_3semillas` | ¿Promediar 3 semillas estabiliza? |
| Ensamble fijo | `ENS_E10S_E7` | E10S + un modelo sin variables espaciales (cobertura de riesgo) |

**Datos de cartografía (robusto):** se descargan del servicio público oficial de la UPRA (derivado del MGN del DANE)
y quedan en caché. Si no hay conexión, se puede subir `geografia.csv`. Antes de usarlos se validan (cantidad,
códigos, coordenadas, áreas y cruce con la densidad del reto). Si la validación falla, las variantes espaciales se
omiten y el notebook sigue.

⚠ Antes de **entregar** una variante espacial, confirmar con los organizadores que la cartografía está permitida.
Regla para cambiar la entrega: mejorar ≥ 0.5 puntos a la referencia sin empeorar en F3.

## 1. Datos, calidad y tratamiento (igual que antes)

In [1]:
import os, zipfile
import numpy as np
import pandas as pd

URL_DATOS = "PEGUE_AQUI_EL_ENLACE_DE_LOS_DATOS"  # enlace de Google Drive al archivo hackathon_datos.zip (se comparte el día del evento)
DATA_DIR = os.environ.get("HACKATHON_DATA", "datos")

if not os.path.exists(os.path.join(DATA_DIR, "entrenamiento.csv")):
    if URL_DATOS:
        import gdown
        import re
        ident = re.search(r"/d/([\w-]+)", URL_DATOS)   # enlace directo: funciona con cualquier versión de gdown
        url = f"https://drive.google.com/uc?id={ident.group(1)}" if ident else URL_DATOS
        gdown.download(url, "hackathon_datos.zip", quiet=False)
    with zipfile.ZipFile("hackathon_datos.zip") as z:
        z.extractall(DATA_DIR)

leer = lambda f: pd.read_csv(os.path.join(DATA_DIR, f), dtype={"DIVIPOLA": str}, parse_dates=["fecha"])
train = leer("entrenamiento.csv")
test_pub = leer("prueba_equipos.csv")     # periodo de la tabla de posiciones en vivo
test_priv = leer("prueba_oculta.csv")     # periodo de la nota oficial
dic = pd.read_csv(os.path.join(DATA_DIR, "diccionario_datos.csv"))
print(train.shape, test_pub.shape, test_priv.shape)

(62814, 25) (20938, 13) (22040, 13)


In [2]:
ESPERADAS = {"entrenamiento": 62_814, "prueba_equipos": 20_938, "prueba_oculta": 22_040}
for nombre, df in {"entrenamiento": train, "prueba_equipos": test_pub, "prueba_oculta": test_priv}.items():
    print(f"{nombre:15s} filas={len(df):,} (esperadas {ESPERADAS[nombre]:,}) · municipios={df.DIVIPOLA.nunique()} · "
          f"llaves repetidas={df.duplicated(['DIVIPOLA', 'fecha']).sum()} · celdas nulas={int(df.isna().sum().sum())}")
nulos = pd.concat({"entrenamiento": train.isna().sum(), "prueba_equipos": test_pub.isna().sum(),
                   "prueba_oculta": test_priv.isna().sum()}, axis=1)
display(nulos[nulos.sum(axis=1) > 0])
fuera = pd.concat([test_pub, test_priv]).ONI.gt(train.ONI.max()).mean()
print(f"ONI de prueba por encima del máximo del entrenamiento: {fuera:.1%} de las filas (los árboles no extrapolan)")

entrenamiento   filas=62,814 (esperadas 62,814) · municipios=1102 · llaves repetidas=0 · celdas nulas=162
prueba_equipos  filas=20,938 (esperadas 20,938) · municipios=1102 · llaves repetidas=0 · celdas nulas=38
prueba_oculta   filas=22,040 (esperadas 22,040) · municipios=1102 · llaves repetidas=0 · celdas nulas=40


,entrenamiento,prueba_equipos,prueba_oculta
densidad_poblacional,12,0.0,0.0
ingresos_tributarios,36,0.0,0.0
inversion_gestion_riesgo,57,19.0,20.0
nbi_pct,57,19.0,20.0


ONI de prueba por encima del máximo del entrenamiento: 20.5% de las filas (los árboles no extrapolan)


In [3]:
COV_ESTATICAS = ["nbi_pct", "elevacion_media", "pendiente_media", "pct_pendiente_alta"]
COV_ANUALES = ["densidad_poblacional", "ingresos_tributarios", "inversion_gestion_riesgo"]
RANGOS = {"densidad_poblacional": (0, None), "ingresos_tributarios": (0, None), "inversion_gestion_riesgo": (0, None),
          "nbi_pct": (0, 100), "elevacion_media": (-50, 6000), "pendiente_media": (0, 90),
          "pct_pendiente_alta": (0, 100), "ONI": (-3.5, 3.5)}
HISTORIAL = ["n_eventos", "familias_desplazadas", "viv_destruidas", "viv_averiadas", "n_movimiento_masa",
             "n_inundacion", "n_vendaval", "n_creciente_subita", "n_avenida_torrencial", "n_otros"]
OBJETIVO = ["tiene_evento", "personas_desplazadas"]


def deduplicar(df):
    df = df.drop_duplicates()
    df = (df.assign(_nulos=df.isna().sum(axis=1)).sort_values(["DIVIPOLA", "fecha", "_nulos"])
            .drop_duplicates(["DIVIPOLA", "fecha"]).drop(columns="_nulos"))
    return df.sort_values(["DIVIPOLA", "fecha"]).reset_index(drop=True)


def preparar(train, test_pub, test_priv):
    partes = {}
    for k, df in {"train": train, "pub": test_pub, "priv": test_priv}.items():
        df = df.copy()
        df["DIVIPOLA"] = df["DIVIPOLA"].astype(str).str.strip().str.zfill(5)
        df["anio"], df["mes"] = df["fecha"].dt.year, df["fecha"].dt.month
        n0 = len(df)
        partes[k] = deduplicar(df)
        if len(partes[k]) != n0:
            print(f"{k}: {n0 - len(partes[k])} filas duplicadas eliminadas")

    tr = partes["train"]
    tr["personas_desplazadas"] = tr["personas_desplazadas"].clip(lower=0)
    tr[HISTORIAL] = tr[HISTORIAL].clip(lower=0).fillna(0)
    viv = tr[["viv_destruidas", "viv_averiadas"]].sum(axis=1)
    afectacion = (tr["personas_desplazadas"].fillna(0) > 0) | (viv > 0)
    corregidas = int(((tr["tiene_evento"] != 1) & afectacion).sum())
    tr["tiene_evento"] = np.where(afectacion, 1, tr["tiene_evento"].fillna(0)).astype(int)
    tr["personas_desplazadas"] = tr["personas_desplazadas"].fillna(0)
    print(f"Objetivo corregido según la definición oficial: {corregidas} filas")

    panel = pd.concat([d.assign(_parte=k, _i=np.arange(len(d))) for k, d in partes.items()], ignore_index=True)
    for c, (lo, hi) in RANGOS.items():
        malo = pd.Series(False, index=panel.index)
        if lo is not None:
            malo |= panel[c] < lo
        if hi is not None:
            malo |= panel[c] > hi
        if malo.any():
            print(f"{c}: {int(malo.sum())} valores imposibles -> nulo")
        panel.loc[malo, c] = np.nan
    faltaba = {c: panel[c].isna() for c in RANGOS if panel[c].isna().any()}

    panel["ONI"] = panel["ONI"].fillna(panel.groupby("fecha")["ONI"].transform("median"))
    for c in COV_ESTATICAS:
        panel[c] = panel[c].fillna(panel.groupby("DIVIPOLA")[c].transform("median"))
    for c in COV_ANUALES:   # el panel está en orden cronológico dentro de cada municipio
        panel[c] = panel[c].fillna(panel.groupby(["DIVIPOLA", "anio"])[c].transform("median"))
        panel[c] = panel.groupby("DIVIPOLA")[c].transform(lambda s: s.ffill().bfill())
    dep = panel["DIVIPOLA"].str[:2]
    for c in COV_ESTATICAS + COV_ANUALES:
        grupos = [dep, panel["anio"]] if c in COV_ANUALES else [dep]
        panel[c] = panel[c].fillna(panel.groupby(grupos)[c].transform("median")).fillna(panel[c].median())
    for c, mascara in faltaba.items():
        panel[c + "_imputado"] = mascara.astype(int)
        print(f"{c}: {int(mascara.sum()):,} valores imputados")
    panel["fase_enso"] = np.select([panel.ONI >= 0.5, panel.ONI <= -0.5], ["El Nino", "La Nina"], "Neutral")

    salida = {k: panel[panel._parte == k].sort_values("_i").drop(columns=["_parte", "_i"]).reset_index(drop=True)
              for k in partes}
    salida["train"]["tiene_evento"] = salida["train"]["tiene_evento"].astype(int)
    for k in ("pub", "priv"):
        salida[k] = salida[k].drop(columns=OBJETIVO + HISTORIAL, errors="ignore")
    return salida["train"], salida["pub"], salida["priv"]


train, test_pub, test_priv = preparar(train, test_pub, test_priv)
print(train.shape, test_pub.shape, test_priv.shape)

Objetivo corregido según la definición oficial: 0 filas
densidad_poblacional: 12 valores imputados
ingresos_tributarios: 36 valores imputados
inversion_gestion_riesgo: 96 valores imputados
nbi_pct: 96 valores imputados
(62814, 29) (20938, 17) (22040, 17)


## 2. Historial de cada municipio (igual que antes)

In [4]:
K = {"dep": 120, "nm": 1102, "dm": 60, "mun": 24, "mm": 3, "mag": 3}
HIST_COLS = ["hist_tasa_mun", "hist_tasa_mes", "hist_tasa_dep", "hist_tasa_dep_mes", "hist_tasa_nac_mes",
             "hist_mag_mun", "hist_esperado_log", "hist_pct_inund", "hist_pct_masa", "hist_reciente_12"]


def _conteos(hist, claves, p):
    return (hist.groupby(claves).agg(**{f"e_{p}": ("tiene_evento", "sum"), f"n_{p}": ("tiene_evento", "size"),
                                         f"lp_{p}": ("logp", "sum")}).reset_index())


def historial_congelado(hist, filas):
    """Historial calculado con todo `hist` (que es pasado respecto a `filas`), aplicado a `filas`."""
    hist = hist.assign(logp=np.log1p(hist["personas_desplazadas"]), dep=hist["DIVIPOLA"].str[:2])
    f = filas.assign(dep=filas["DIVIPOLA"].str[:2])
    for claves, p in [(["DIVIPOLA"], "mun"), (["DIVIPOLA", "mes"], "mm"), (["dep"], "dep"),
                      (["dep", "mes"], "dm"), (["mes"], "nm")]:
        f = f.merge(_conteos(hist, claves, p), on=claves, how="left")
    c = f.fillna({col: 0 for col in f.columns if col[:2] in ("e_", "n_") or col.startswith("lp_")})
    tasa_nac = hist["tiene_evento"].mean()
    tasa_nm = (c.e_nm + K["nm"] * tasa_nac) / (c.n_nm + K["nm"])
    tasa_dep = (c.e_dep + K["dep"] * tasa_nac) / (c.n_dep + K["dep"])
    tasa_dm = (c.e_dm + K["dm"] * tasa_dep * tasa_nm / tasa_nac) / (c.n_dm + K["dm"])
    tasa_mun = (c.e_mun + K["mun"] * tasa_dep) / (c.n_mun + K["mun"])
    tasa_mm = (c.e_mm + K["mm"] * tasa_mun * tasa_dm / tasa_dep) / (c.n_mm + K["mm"])
    mag_nac = hist["logp"].sum() / max(hist["tiene_evento"].sum(), 1)
    mag_dep = (c.lp_dep + 5 * mag_nac) / (c.e_dep + 5)
    mag_mun = (c.lp_mun + K["mag"] * mag_dep) / (c.e_mun + K["mag"])

    tipos = hist.groupby("DIVIPOLA")[["n_inundacion", "n_movimiento_masa", "n_eventos"]].sum()
    reciente = hist[hist["fecha"] > hist["fecha"].max() - pd.DateOffset(months=12)].groupby("DIVIPOLA")["tiene_evento"].mean()
    salida = filas.copy()
    salida["hist_tasa_mun"] = tasa_mun.to_numpy()
    salida["hist_tasa_mes"] = tasa_mm.to_numpy()
    salida["hist_tasa_dep"] = tasa_dep.to_numpy()
    salida["hist_tasa_dep_mes"] = tasa_dm.to_numpy()
    salida["hist_tasa_nac_mes"] = tasa_nm.to_numpy()
    salida["hist_mag_mun"] = mag_mun.to_numpy()
    salida["hist_esperado_log"] = (tasa_mm * mag_mun).to_numpy()   # E[log1p(personas)] según la historia
    salida["hist_pct_inund"] = filas["DIVIPOLA"].map((tipos.n_inundacion + 1) / (tipos.n_eventos + 3)).fillna(1 / 3).to_numpy()
    salida["hist_pct_masa"] = filas["DIVIPOLA"].map((tipos.n_movimiento_masa + 1) / (tipos.n_eventos + 3)).fillna(1 / 3).to_numpy()
    salida["hist_reciente_12"] = filas["DIVIPOLA"].map(reciente).fillna(tasa_nac).to_numpy()
    return salida


def historial_entrenamiento(hist):
    """Filas de entrenamiento: para cada año Y (desde el segundo), historial con los años anteriores a Y."""
    anios = sorted(hist["anio"].unique())
    return pd.concat([historial_congelado(hist[hist.anio < a], hist[hist.anio == a]) for a in anios[1:]],
                     ignore_index=True)

## 3. Cartografía municipal: carga robusta y validación
1. Si existe la caché `cartografia_municipios.geojson`, se usa.
2. Si no, se descarga del servicio público de la UPRA (capa de municipios generalizada, derivada del MGN del DANE).
   La simplificación de los polígonos la hace el servidor.
3. Si la descarga falla, se usa `geografia.csv` si lo subieron (está en `data/externos/procesados/` del repo).
4. Se calculan el centroide y el área de cada municipio y se validan. Del dataset solo se cruza la densidad, para
   verificar que población = densidad × área tenga sentido.

In [5]:
import json
import math
import urllib.parse
import urllib.request

CARTO_SERVICIO = ("https://geoservicios.upra.gov.co/arcgis/rest/services/referencia_geografica/"
                  "municipios_generalizada/MapServer/0/query")
CARTO_GEOJSON = "cartografia_municipios.geojson"   # caché local, se crea en la primera corrida
CARTO_CSV = "geografia.csv"                        # alternativa sin internet
KM_POR_GRADO_LAT, KM_POR_GRADO_LON = 110.574, 111.320


def descargar_cartografia(tolerancia=0.002):
    parametros = {"where": "1=1", "outFields": "cod_dane_mpio,municipio,departamen", "outSR": 4326,
                  "maxAllowableOffset": tolerancia, "geometryPrecision": 4, "resultRecordCount": 2000, "f": "geojson"}
    with urllib.request.urlopen(f"{CARTO_SERVICIO}?{urllib.parse.urlencode(parametros)}", timeout=120) as r:
        return json.loads(r.read().decode("utf-8"))


def centroide_y_area(geometria):
    """Centroide del polígono exterior más grande y área en km² (proyección local equirectangular)."""
    poligonos = [geometria["coordinates"]] if geometria["type"] == "Polygon" else geometria["coordinates"]
    area_km2, mejor, mejor_area = 0.0, None, -1.0
    for poligono in poligonos:
        lat_media = sum(y for _, y in poligono[0]) / len(poligono[0])
        escala = KM_POR_GRADO_LAT * KM_POR_GRADO_LON * math.cos(math.radians(lat_media))
        for k, anillo in enumerate(poligono):
            a = cx = cy = 0.0
            for (x0, y0), (x1, y1) in zip(anillo, anillo[1:] + anillo[:1]):
                cruz = x0 * y1 - x1 * y0
                a, cx, cy = a + cruz, cx + (x0 + x1) * cruz, cy + (y0 + y1) * cruz
            a /= 2
            area_km2 += (abs(a) if k == 0 else -abs(a)) * escala
            if k == 0 and a != 0 and abs(a) > mejor_area:
                mejor_area, mejor = abs(a), (cx / (6 * a), cy / (6 * a))
    return mejor, area_km2


def tabla_desde_geojson(geojson):
    filas = []
    for f in geojson["features"]:
        if f.get("geometry"):
            (lon, lat), area = centroide_y_area(f["geometry"])
            filas.append({"DIVIPOLA": str(f["properties"]["cod_dane_mpio"]).zfill(5),
                          "lat": round(lat, 4), "lon": round(lon, 4), "area_km2": round(area, 1)})
    return pd.DataFrame(filas).drop_duplicates("DIVIPOLA").reset_index(drop=True)


def cargar_cartografia():
    geojson, origen = None, None
    if os.path.exists(CARTO_GEOJSON):
        with open(CARTO_GEOJSON, encoding="utf-8") as f:
            geojson, origen = json.load(f), f"caché local ({CARTO_GEOJSON})"
    else:
        try:
            geojson, origen = descargar_cartografia(), "servicio público de la UPRA (MGN del DANE)"
            with open(CARTO_GEOJSON, "w", encoding="utf-8") as f:
                json.dump(geojson, f)
        except Exception as error:
            print(f"⚠ No se pudo descargar la cartografía: {error}")
    if geojson is not None:
        return tabla_desde_geojson(geojson), geojson, origen
    if os.path.exists(CARTO_CSV):
        tabla = pd.read_csv(CARTO_CSV, dtype={"DIVIPOLA": str})[["DIVIPOLA", "lat", "lon", "area_km2"]]
        return tabla, None, f"archivo subido ({CARTO_CSV}; sin polígonos: no hay vecindad por frontera)"
    return None, None, "no disponible"


def validar_cartografia(tabla, datos):
    t = tabla.set_index("DIVIPOLA")
    codigos = set(datos["DIVIPOLA"])
    faltan = sorted(codigos - set(tabla["DIVIPOLA"]))
    dens = datos.loc[(datos["DIVIPOLA"] == "11001") & (datos["anio"] == datos["anio"].max()), "densidad_poblacional"]
    pob_bogota = float(dens.iloc[0] * t["area_km2"].get("11001", np.nan)) if len(dens) else float("nan")
    bloqueantes = [
        ("al menos 1,100 municipios", len(tabla) >= 1100),
        ("códigos DIVIPOLA únicos y de 5 dígitos",
         tabla["DIVIPOLA"].is_unique and tabla["DIVIPOLA"].str.fullmatch(r"\d{5}").all()),
        ("centroides dentro de Colombia",
         tabla["lat"].between(-4.5, 13.6).all() and tabla["lon"].between(-82.0, -66.8).all()),
        ("áreas positivas", (tabla["area_km2"] > 0).all()),
        ("área de Bogotá ≈ 1,636 km² (±5 %)", abs(t["area_km2"].get("11001", 0) / 1636 - 1) < 0.05),
    ]
    informativas = [
        (f"población de Bogotá = densidad × área ≈ 7–8 millones (da {pob_bogota / 1e6:.1f} M)", 5e6 < pob_bogota < 1e7),
        (f"cubre los {len(codigos):,} municipios del reto (faltan {len(faltan)}: {faltan[:5]})", not faltan),
    ]
    for texto, ok in bloqueantes + informativas:
        print(("✅ " if ok else "❌ ") + texto)
    return all(ok for _, ok in bloqueantes)


GEO_TABLA, GEO_JSON, origen = cargar_cartografia()
USAR_ESPACIAL = GEO_TABLA is not None and validar_cartografia(GEO_TABLA, train)
print(f"Cartografía: {origen} → variantes espaciales {'ACTIVADAS' if USAR_ESPACIAL else 'DESACTIVADAS'}")

✅ al menos 1,100 municipios
✅ códigos DIVIPOLA únicos y de 5 dígitos
✅ centroides dentro de Colombia
✅ áreas positivas
✅ área de Bogotá ≈ 1,636 km² (±5 %)
✅ población de Bogotá = densidad × área ≈ 7–8 millones (da 7.8 M)
✅ cubre los 1,102 municipios del reto (faltan 0: [])
Cartografía: servicio público de la UPRA (MGN del DANE) → variantes espaciales ACTIVADAS


## 4. Vecindades
Cuatro definiciones de "vecino", todas con la **misma historia que la fila** (solo pasado, sin fuga):
* `k3`, `k5`, `k10`: los 3, 5 o 10 municipios con el centroide más cercano.
* `k10p`: los 10 más cercanos, ponderados por la inversa de la distancia.
* `frontera`: los que comparten límite (polígonos que se tocan; requiere `shapely`, que Colab trae). Las islas usan
  sus 3 más cercanos.

Para cada definición se calcula el promedio de los vecinos de la tasa histórica (global y del mes), la magnitud típica
y la tasa reciente. Además, el **suavizado espacial**: la tasa del municipio encogida hacia la de sus 5 vecinos.

In [6]:
def pares_knn(tabla, k, ponderado=False):
    lat, lon = np.radians(tabla["lat"].to_numpy()), np.radians(tabla["lon"].to_numpy())
    a = (np.sin((lat[:, None] - lat[None, :]) / 2) ** 2
         + np.cos(lat)[:, None] * np.cos(lat)[None, :] * np.sin((lon[:, None] - lon[None, :]) / 2) ** 2)
    distancia = 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    np.fill_diagonal(distancia, np.inf)
    cercanos = np.argsort(distancia, axis=1)[:, :k]
    km = np.take_along_axis(distancia, cercanos, axis=1) * 6371
    peso = 1 / (km + 1) if ponderado else np.ones_like(km)
    codigos = tabla["DIVIPOLA"].to_numpy()
    return pd.DataFrame({"DIVIPOLA": np.repeat(codigos, k), "vecino": codigos[cercanos.ravel()], "peso": peso.ravel()})


def pares_frontera(geojson, respaldo):
    try:
        from shapely.geometry import shape
        from shapely.strtree import STRtree
    except ImportError:
        print("⚠ shapely no está disponible: se omite la vecindad por frontera")
        return None
    feats = [f for f in geojson["features"] if f.get("geometry")]
    codigos = np.array([str(f["properties"]["cod_dane_mpio"]).zfill(5) for f in feats])
    formas = [shape(f["geometry"]).buffer(0.002) for f in feats]      # tolera la simplificación de los límites
    i, j = STRtree(formas).query(formas, predicate="intersects")
    distintos = i != j
    pares = pd.DataFrame({"DIVIPOLA": codigos[i[distintos]], "vecino": codigos[j[distintos]], "peso": 1.0}).drop_duplicates()
    islas = sorted(set(codigos) - set(pares["DIVIPOLA"]))
    return pd.concat([pares, respaldo[respaldo["DIVIPOLA"].isin(islas)]], ignore_index=True)


VECINDADES = {}
if USAR_ESPACIAL:
    VECINDADES = {"k5": pares_knn(GEO_TABLA, 5), "k3": pares_knn(GEO_TABLA, 3), "k10": pares_knn(GEO_TABLA, 10),
                  "k10p": pares_knn(GEO_TABLA, 10, ponderado=True)}
    if GEO_JSON is not None:
        frontera = pares_frontera(GEO_JSON, VECINDADES["k3"])
        if frontera is not None:
            VECINDADES["frontera"] = frontera
    for nombre, pares in VECINDADES.items():
        print(f"vecindad {nombre:8s}: {pares.groupby('DIVIPOLA').size().mean():.1f} vecinos en promedio")
    train, test_pub, test_priv = (d.merge(GEO_TABLA, on="DIVIPOLA", how="left") for d in (train, test_pub, test_priv))

VEC_COLS = {"hist_tasa_mun": "tasa", "hist_tasa_mes": "tasa_mes", "hist_mag_mun": "mag", "hist_reciente_12": "reciente"}
MESES = pd.DataFrame({"mes": range(1, 13)})


def agregar_vecindad(hist, filas):
    if not USAR_ESPACIAL:
        return filas
    grilla = GEO_TABLA[["DIVIPOLA"]].merge(MESES, how="cross")
    t = historial_congelado(hist, grilla)[["DIVIPOLA", "mes"] + list(VEC_COLS)].rename(columns={"DIVIPOLA": "vecino"})
    salida = filas
    for nombre, pares in VECINDADES.items():
        v = pares.merge(t, on="vecino")
        for c in VEC_COLS:
            v[c] = v[c] * v["peso"]
        g = v.groupby(["DIVIPOLA", "mes"])[list(VEC_COLS) + ["peso"]].sum()
        g = g[list(VEC_COLS)].div(g["peso"], axis=0)
        g.columns = [f"vec_{nombre}_{VEC_COLS[c]}" for c in VEC_COLS]
        salida = salida.merge(g.reset_index(), on=["DIVIPOLA", "mes"], how="left")
    # Suavizado espacial: tasa del municipio encogida hacia la de sus 5 vecinos (en lugar del departamento)
    conteo = hist.groupby("DIVIPOLA")["tiene_evento"].agg(["sum", "size"])
    e = salida["DIVIPOLA"].map(conteo["sum"]).fillna(0).to_numpy()
    n = salida["DIVIPOLA"].map(conteo["size"]).fillna(0).to_numpy()
    previa = salida["vec_k5_tasa"].fillna(hist["tiene_evento"].mean()).to_numpy()
    salida["hist_tasa_mun_espacial"] = (e + K["mun"] * previa) / (n + K["mun"])
    return salida


def historial_completo(hist, filas):
    return agregar_vecindad(hist, historial_congelado(hist, filas))


def historial_entrenamiento_completo(hist):
    anios = sorted(hist["anio"].unique())
    return pd.concat([historial_completo(hist[hist.anio < a], hist[hist.anio == a]) for a in anios[1:]],
                     ignore_index=True)

vecindad k5      : 5.0 vecinos en promedio
vecindad k3      : 3.0 vecinos en promedio
vecindad k10     : 10.0 vecinos en promedio
vecindad k10p    : 10.0 vecinos en promedio
vecindad frontera: 5.9 vecinos en promedio


## 5. Variables

In [7]:
ONI_SERIE = pd.concat([train, test_pub, test_priv])[["fecha", "ONI"]].groupby("fecha")["ONI"].median()


def features_base(df):
    X = pd.DataFrame(index=df.index)
    for c in COV_ANUALES:
        X["log_" + c] = np.log1p(df[c].clip(lower=0))
    X["ingresos_rango_anio"] = df.groupby("anio")["ingresos_tributarios"].rank(pct=True)
    for c in COV_ESTATICAS + ["ONI"] + HIST_COLS:
        X[c] = df[c]
    for c in [c for c in df.columns if c.endswith("_imputado")]:
        X[c] = df[c]
    for k in (1, 2, 3):
        X[f"ONI_rezago_{k}"] = (df["fecha"] - pd.DateOffset(months=k)).map(ONI_SERIE).to_numpy()
    X["mes"] = df["mes"]
    X["mes_sin"], X["mes_cos"] = np.sin(2 * np.pi * df["mes"] / 12), np.cos(2 * np.pi * df["mes"] / 12)
    X["departamento"] = df["DIVIPOLA"].str[:2].astype(int)
    X["oni_x_inundacion"] = df["ONI"] * df["hist_pct_inund"]
    X["oni_x_tasa"] = df["ONI"] * df["hist_tasa_mun"]
    return X


def features(df, cfg):
    X = features_base(df)
    if cfg.get("espacial"):
        grupos, v = cfg["grupos"], cfg["vecindad"]
        if "ubicacion" in grupos:
            X["lat"], X["lon"] = df["lat"], df["lon"]
        if "poblacion_area" in grupos:
            X["log_area"] = np.log1p(df["area_km2"])
            X["log_poblacion"] = np.log1p(df["densidad_poblacional"].clip(lower=0) * df["area_km2"])
        if "vecinos" in grupos:
            X["hist_vecinos_tasa"], X["hist_vecinos_tasa_mes"] = df[f"vec_{v}_tasa"], df[f"vec_{v}_tasa_mes"]
            if cfg.get("vecinos_plus"):
                X["hist_vecinos_mag"], X["hist_vecinos_reciente"] = df[f"vec_{v}_mag"], df[f"vec_{v}_reciente"]
        if cfg.get("suavizado_espacial"):
            X["hist_tasa_mun_espacial"] = df["hist_tasa_mun_espacial"]
    return X

## 6. Métricas oficiales y validador de la plantilla

In [8]:
from sklearn.metrics import roc_auc_score

def metricas(pred, verdad):
    m = verdad.merge(pred, on=["DIVIPOLA", "fecha"])
    y = np.log1p(m["personas_desplazadas"])
    est = np.log1p(m["personas_desplazadas_estimadas"])
    lo, hi = np.log1p(m["personas_desplazadas_q10"]), np.log1p(m["personas_desplazadas_q90"])
    wink = (hi - lo) + 10 * np.clip(lo - y, 0, None) + 10 * np.clip(y - hi, 0, None)   # alpha = 0.2
    return {"auc": roc_auc_score(m["tiene_evento"], m["prob_evento"]),
            "rmse_log": float(np.sqrt(np.mean((y - est) ** 2))), "winkler": float(wink.mean())}

def puntos(met, nula):
    """Parte automática de la nota (máx. 75 puntos), como la tabla de posiciones."""
    s_auc = max(0, (met["auc"] - 0.5) / 0.5)
    s_rmse = min(1, max(0, 1 - met["rmse_log"] / nula["rmse_log"]))
    s_wink = min(1, max(0, 1 - met["winkler"] / nula["winkler"]))
    return 100 * (0.50 * s_auc + 0.20 * s_rmse + 0.05 * s_wink)


def prediccion_nula(filas):
    return filas[["DIVIPOLA", "fecha"]].assign(prob_evento=0.5, personas_desplazadas_estimadas=0.0,
                                               personas_desplazadas_q10=0.0, personas_desplazadas_q90=0.0)


def winkler_con_factor(pred, verdad, factor):
    m = verdad.merge(pred, on=["DIVIPOLA", "fecha"])
    y, lo = np.log1p(m["personas_desplazadas"]), np.log1p(m["personas_desplazadas_q10"])
    hi = np.maximum(lo, np.log1p(m["personas_desplazadas_q90"]) * factor)
    return float(((hi - lo) + 10 * np.clip(lo - y, 0, None) + 10 * np.clip(y - hi, 0, None)).mean())

In [9]:
COLUMNAS = ["DIVIPOLA", "fecha", "prob_evento", "personas_desplazadas_estimadas",
            "personas_desplazadas_q10", "personas_desplazadas_q90"]

def validar_entrega(e, test_pub=test_pub, test_priv=test_priv):
    errores = []
    faltan = [c for c in COLUMNAS if c not in e.columns]
    if faltan:
        return print("❌ Faltan columnas:", faltan)
    claves = pd.concat([test_pub, test_priv])[["DIVIPOLA", "fecha"]].assign(fecha=lambda d: pd.to_datetime(d.fecha))
    e2 = e.assign(DIVIPOLA=e.DIVIPOLA.astype(str).str.zfill(5), fecha=pd.to_datetime(e.fecha))
    if e2.duplicated(["DIVIPOLA", "fecha"]).any():
        errores.append("hay filas DIVIPOLA-fecha duplicadas")
    faltantes = len(claves.merge(e2, on=["DIVIPOLA", "fecha"], how="left", indicator=True).query("_merge == 'left_only'"))
    if faltantes:
        errores.append(f"faltan {faltantes} filas de las {len(claves)} esperadas")
    num = e2[COLUMNAS[2:]].apply(pd.to_numeric, errors="coerce")
    if num.isna().any().any():
        errores.append("hay valores vacíos o no numéricos")
    if not num.prob_evento.between(0, 1).all():
        errores.append("prob_evento debe estar entre 0 y 1")
    if (num[COLUMNAS[3:]] < 0).any().any():
        errores.append("las columnas de personas deben ser >= 0")
    if (num.personas_desplazadas_q10 > num.personas_desplazadas_q90).any():
        errores.append("personas_desplazadas_q10 debe ser <= personas_desplazadas_q90")
    print("✅ Entrega válida:", len(e), "filas" if not errores else "", *["\n❌ " + x for x in errores])
    return not errores

## 7. Variantes y modelos

In [10]:
import shutil
import time

import lightgbm as lgb

NOMBRE_EQUIPO = "AICOLOMBIA"
LGBM_BASE = dict(learning_rate=0.05, num_leaves=31, min_child_samples=100, colsample_bytree=0.8, subsample=0.8,
                 subsample_freq=1, reg_lambda=1.0, deterministic=True, force_row_wise=True, verbose=-1)
ES_MESES, ES_PACIENCIA, FACTOR_ARBOLES, MAX_ARBOLES = 12, 100, 1.15, 3000

SIMPLE = dict(num_leaves=15, min_child_samples=200)
E10S = dict(espacial=True, grupos=("ubicacion", "poblacion_area", "vecinos"), vecindad="k5", vecinos_plus=False,
            suavizado_espacial=False, arboles_p=100, arboles_mag=None, arboles_q=None, lgbm=SIMPLE, semillas=(42,))
E7 = dict(espacial=False, arboles_p=None, arboles_mag=None, arboles_q=None, lgbm=dict(learning_rate=0.02),
          semillas=(42,))
VARIANTES = {
    "E10S": E10S,
    "E10S_sin_poblacion": {**E10S, "grupos": ("ubicacion", "vecinos")},
    "E10S_sin_ubicacion": {**E10S, "grupos": ("poblacion_area", "vecinos")},
    "E10S_sin_vecinos": {**E10S, "grupos": ("ubicacion", "poblacion_area")},
    "E10S_k3": {**E10S, "vecindad": "k3"},
    "E10S_k10": {**E10S, "vecindad": "k10"},
    "E10S_k10_ponderado": {**E10S, "vecindad": "k10p"},
    "E10S_frontera": {**E10S, "vecindad": "frontera"},
    "E10S_vecinos_plus": {**E10S, "vecinos_plus": True},
    "E10S_suavizado_espacial": {**E10S, "suavizado_espacial": True},
    "E10S_arboles_50": {**E10S, "arboles_p": 50},
    "E10S_arboles_150": {**E10S, "arboles_p": 150},
    "E10S_hojas_7": {**E10S, "lgbm": {**SIMPLE, "num_leaves": 7}},
    "E10S_lento": {**E10S, "arboles_p": 250, "lgbm": {**SIMPLE, "learning_rate": 0.02}},
    "E10S_todo_simple": {**E10S, "arboles_mag": 100, "arboles_q": 100},
    "E10S_3semillas": {**E10S, "semillas": (42, 7, 2026)},
    "E7_lento": E7,
}
if not USAR_ESPACIAL:
    VARIANTES = {"E7_lento": E7}
elif "frontera" not in VECINDADES:
    VARIANTES.pop("E10S_frontera")

DESCRIPCION = {
    "E10S": "Referencia: E10_espacial_simple de la ronda 2",
    "E10S_sin_poblacion": "E10S sin población ni área",
    "E10S_sin_ubicacion": "E10S sin latitud ni longitud",
    "E10S_sin_vecinos": "E10S sin la tasa de los vecinos",
    "E10S_k3": "Vecinos: 3 más cercanos",
    "E10S_k10": "Vecinos: 10 más cercanos",
    "E10S_k10_ponderado": "Vecinos: 10 más cercanos ponderados por distancia",
    "E10S_frontera": "Vecinos: los que comparten frontera",
    "E10S_vecinos_plus": "Agrega magnitud y tasa reciente de los vecinos",
    "E10S_suavizado_espacial": "Agrega la tasa del municipio encogida hacia sus vecinos",
    "E10S_arboles_50": "Clasificador con 50 árboles",
    "E10S_arboles_150": "Clasificador con 150 árboles",
    "E10S_hojas_7": "Árboles de 7 hojas",
    "E10S_lento": "Tasa de aprendizaje 0.02 con 250 árboles",
    "E10S_todo_simple": "Magnitud y cuantiles también simples (100 árboles)",
    "E10S_3semillas": "Promedio de 3 semillas",
    "E7_lento": "Referencia sin espaciales: E7_lento de la ronda 2",
    "ENS_E10S_E7": "Promedio fijo de E10S y E7_lento",
}


def ajustar(X, y, fechas, clase, cfg, n_arboles, semilla, **extra):
    params = {**LGBM_BASE, **cfg["lgbm"], **extra, "random_state": semilla}
    if n_arboles is not None:
        return clase(**params, n_estimators=n_arboles).fit(X, y)
    meses = np.sort(fechas.unique())
    corte = meses[-min(ES_MESES, max(3, len(meses) // 4))]
    a, b = (fechas < corte).to_numpy(), (fechas >= corte).to_numpy()
    m = clase(**params, n_estimators=MAX_ARBOLES).fit(X[a], y[a], eval_set=[(X[b], y[b])],
                                                     callbacks=[lgb.early_stopping(ES_PACIENCIA, verbose=False)])
    n = max(50, int(round((m.best_iteration_ or MAX_ARBOLES) * FACTOR_ARBOLES)))
    return clase(**params, n_estimators=n).fit(X, y)


def entrenar(filas, cfg):
    X, fechas = features(filas, cfg), filas["fecha"]
    y_ev, y_log = filas["tiene_evento"], np.log1p(filas["personas_desplazadas"])
    conjuntos = []
    for s in cfg["semillas"]:
        m = {"p": ajustar(X, y_ev, fechas, lgb.LGBMClassifier, cfg, cfg["arboles_p"], s, objective="binary", metric="auc"),
             "e": ajustar(X, y_log, fechas, lgb.LGBMRegressor, cfg, cfg["arboles_mag"], s, objective="regression", metric="l2")}
        for q in (0.1, 0.9):
            m[q] = ajustar(X, y_log, fechas, lgb.LGBMRegressor, cfg, cfg["arboles_q"], s, objective="quantile",
                           alpha=q, metric="quantile")
        conjuntos.append(m)
    return conjuntos, None


def predecir_log(conjuntos, filas, cfg):
    X = features(filas, cfg)
    p = filas[["DIVIPOLA", "fecha"]].copy()
    p["prob"] = np.mean([m["p"].predict_proba(X)[:, 1] for m in conjuntos], axis=0)
    p["mag"] = np.mean([np.clip(m["e"].predict(X), 0, None) for m in conjuntos], axis=0)
    p["lo"] = np.mean([np.clip(m[0.1].predict(X), 0, None) for m in conjuntos], axis=0)
    p["hi"] = np.mean([np.clip(m[0.9].predict(X), 0, None) for m in conjuntos], axis=0)
    return p


def a_entrega(cruda, factor_q90=1.0):
    p = cruda[["DIVIPOLA", "fecha"]].copy()
    p["prob_evento"] = cruda["prob"].to_numpy()
    p["personas_desplazadas_estimadas"] = np.expm1(cruda["mag"].to_numpy())
    hi = cruda["hi"].to_numpy() * factor_q90
    p["personas_desplazadas_q10"] = np.expm1(np.minimum(cruda["lo"].to_numpy(), hi))
    p["personas_desplazadas_q90"] = np.expm1(np.maximum(cruda["lo"].to_numpy(), hi))
    return p

## 8. Correr la ronda 3
El historial con todas las vecindades se calcula una sola vez por corte. Por variante: validación F1–F3, factor de
`q90`, modelo final y CSV validado en `entregas_ronda3/`.

In [11]:
CORTES = [("F1", "2020-09-01", "2020-10-01", "2021-09-01"),
          ("F2", "2021-09-01", "2021-10-01", "2022-09-01"),
          ("F3", "2020-09-01", "2021-10-01", "2022-09-01")]
FACTORES = [0.7, 0.8, 0.9, 1.0, 1.1, 1.25, 1.4]
CARPETA = "entregas_ronda3"

t0 = time.time()
PREP = {}
for nombre, fin_hist, ini_val, fin_val in CORTES:
    hist = train[train.fecha <= fin_hist]
    va = train[(train.fecha >= ini_val) & (train.fecha <= fin_val)]
    PREP[nombre] = (historial_entrenamiento_completo(hist), historial_completo(hist, va), va)
PRUEBA = pd.concat([test_pub, test_priv], ignore_index=True)
FINAL = (historial_entrenamiento_completo(train), historial_completo(train, PRUEBA))
print(f"Historial y vecindad preparados en {time.time() - t0:.0f}s")


def evaluar(crudas_por_corte):
    """Métricas por corte, factor de q90 y cobertura a partir de predicciones crudas."""
    por_corte, factores = {}, []
    for corte, cruda in crudas_por_corte.items():
        va = PREP[corte][2]
        pred = a_entrega(cruda)
        met, nula = metricas(pred, va), metricas(prediccion_nula(va), va)
        winklers = {f: winkler_con_factor(pred, va, f) for f in FACTORES}
        factores.append(min(winklers, key=winklers.get))
        m = va.merge(pred, on=["DIVIPOLA", "fecha"])
        cobertura = float(((m.personas_desplazadas >= m.personas_desplazadas_q10) &
                           (m.personas_desplazadas <= m.personas_desplazadas_q90)).mean())
        por_corte[corte] = {**met, "puntos": puntos(met, nula), "cobertura": cobertura}
    return pd.DataFrame(por_corte).T, float(np.median(factores))


def guardar(variante, cruda_final, factor):
    entrega = a_entrega(cruda_final, factor)
    entrega["fecha"] = pd.to_datetime(entrega["fecha"]).dt.strftime("%Y-%m-%d")
    entrega = entrega[COLUMNAS]
    if not validar_entrega(entrega):
        return "INVÁLIDA"
    archivo = f"{CARPETA}/entrega_{NOMBRE_EQUIPO}_{variante}.csv"
    entrega.to_csv(archivo, index=False)
    return archivo


os.makedirs(CARPETA, exist_ok=True)
CRUDAS, CRUDAS_FINAL, resumen, detalle = {}, {}, [], []


def registrar(variante, d, factor, archivo, segundos):
    resumen.append({"ronda": 3, "variante": variante, "descripcion": DESCRIPCION[variante],
                    "puntos_prom": d.puntos.mean(), "puntos_desv": d.puntos.std(),
                    "puntos_F1": d.loc["F1", "puntos"], "puntos_F2": d.loc["F2", "puntos"],
                    "puntos_F3": d.loc["F3", "puntos"], "auc_prom": d.auc.mean(), "rmse_log_prom": d.rmse_log.mean(),
                    "winkler_prom": d.winkler.mean(), "cobertura_prom": d.cobertura.mean(), "factor_q90": factor,
                    "archivo": archivo})
    for corte in d.index:
        detalle.append({"ronda": 3, "variante": variante, "corte": corte, **d.loc[corte].to_dict()})
    print(f"{variante:22s} puntos {d.puntos.mean():5.1f} ± {d.puntos.std():.1f} · F2 {d.loc['F2', 'puntos']:5.1f} "
          f"· F3 {d.loc['F3', 'puntos']:5.1f} · {segundos:.0f}s")


for variante, cambios in VARIANTES.items():
    t1 = time.time()
    cfg = cambios
    CRUDAS[variante] = {}
    for corte, (filas_hist, filas_va, va) in PREP.items():
        modelos_cv, _ = entrenar(filas_hist, cfg)
        CRUDAS[variante][corte] = predecir_log(modelos_cv, filas_va, cfg)
    d, factor = evaluar(CRUDAS[variante])
    modelos_fin, _ = entrenar(FINAL[0], cfg)
    CRUDAS_FINAL[variante] = predecir_log(modelos_fin, FINAL[1], cfg)
    registrar(variante, d, factor, guardar(variante, CRUDAS_FINAL[variante], factor), time.time() - t1)

# Ensamble fijo (no elegido mirando los resultados): E10S + E7_lento
if {"E10S", "E7_lento"} <= set(CRUDAS):
    promedio = lambda partes: partes[0][["DIVIPOLA", "fecha"]].assign(
        **{c: np.mean([p[c].to_numpy() for p in partes], axis=0) for c in ["prob", "mag", "lo", "hi"]})
    crudas_ens = {corte: promedio([CRUDAS[v][corte] for v in ("E10S", "E7_lento")]) for corte in PREP}
    d, factor = evaluar(crudas_ens)
    registrar("ENS_E10S_E7", d, factor,
              guardar("ENS_E10S_E7", promedio([CRUDAS_FINAL[v] for v in ("E10S", "E7_lento")]), factor), 0)
print(f"Total: {(time.time() - t0) / 60:.1f} min")

Historial y vecindad preparados en 6s


c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is depreca

✅ Entrega válida: 42978 filas
E10S                   puntos  22.0 ± 2.5 · F2  24.6 · F3  19.6 · 11s


c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is depreca

✅ Entrega válida: 42978 filas
E10S_sin_poblacion     puntos  21.6 ± 2.9 · F2  24.6 · F3  18.8 · 8s


c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is depreca

... (23 outputs left out)

c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is depreca

✅ Entrega válida: 42978 filas
E10S_3semillas         puntos  22.0 ± 2.8 · F2  24.8 · F3  19.3 · 31s


c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is depreca

✅ Entrega válida: 42978 filas
E7_lento               puntos  21.7 ± 2.8 · F2  24.2 · F3  18.7 · 32s
✅ Entrega válida: 42978 filas
ENS_E10S_E7            puntos  22.1 ± 2.6 · F2  24.7 · F3  19.5 · 0s
Total: 3.8 min


## 9. Resultados: ronda 3 frente a las rondas anteriores

In [12]:
# Referencias de la ronda 2 (Colab), por corte
RONDA2 = pd.DataFrame([
    ("E9_ensamble_top3", 0.700, 0.730, 0.677, 1.351, 1.258, 1.265, 3.505, 3.269, 3.459, 21.975, 24.909, 19.242),
    ("E10_espacial_simple", 0.696, 0.731, 0.677, 1.351, 1.259, 1.263, 3.511, 3.316, 3.454, 21.596, 24.938, 19.348),
    ("E7_lento", 0.701, 0.723, 0.672, 1.351, 1.260, 1.268, 3.499, 3.262, 3.475, 22.057, 24.199, 18.697),
    ("E10_espacial", 0.693, 0.728, 0.671, 1.353, 1.261, 1.267, 3.520, 3.251, 3.468, 21.282, 24.670, 18.622),
    ("E6_simple", 0.693, 0.721, 0.668, 1.351, 1.259, 1.265, 3.500, 3.295, 3.441, 21.299, 23.977, 18.416),
    ("V1_early_stopping", 0.679, 0.722, 0.647, 1.353, 1.263, 1.268, 3.513, 3.248, 3.482, 19.872, 24.070, 16.250),
], columns=["variante", "auc_F1", "auc_F2", "auc_F3", "rmse_F1", "rmse_F2", "rmse_F3",
            "wink_F1", "wink_F2", "wink_F3", "puntos_F1", "puntos_F2", "puntos_F3"])
P = ["puntos_F1", "puntos_F2", "puntos_F3"]
anteriores = RONDA2.assign(ronda=2, descripcion="(ronda 2)", archivo="zip de la ronda 2",
                           puntos_prom=RONDA2[P].mean(axis=1), puntos_desv=RONDA2[P].std(axis=1),
                           auc_prom=RONDA2[["auc_F1", "auc_F2", "auc_F3"]].mean(axis=1),
                           rmse_log_prom=RONDA2[["rmse_F1", "rmse_F2", "rmse_F3"]].mean(axis=1),
                           winkler_prom=RONDA2[["wink_F1", "wink_F2", "wink_F3"]].mean(axis=1))

r3 = pd.DataFrame(resumen)
for nuevo, viejo in [("E10S", "E10_espacial_simple"), ("E7_lento", "E7_lento")]:
    if nuevo in set(r3.variante):
        a = r3.set_index("variante").loc[nuevo, "puntos_prom"]
        b = anteriores.set_index("variante").loc[viejo, "puntos_prom"]
        print(f"Reproducibilidad {nuevo}: ronda 2 {b:.2f} vs ronda 3 {a:.2f} "
              + ("✅" if abs(a - b) < 0.05 else "⚠ difiere: normal si cambió el entorno, la versión de LightGBM o la cartografía"))

referencia = "E10S" if "E10S" in set(r3.variante) else "E7_lento"
ref = r3.set_index("variante").loc[referencia]
todas = pd.concat([r3, anteriores[~anteriores.variante.isin(["E7_lento"])]], ignore_index=True)
todas["delta_vs_V1"] = todas["puntos_prom"] - anteriores.set_index("variante").loc["V1_early_stopping", "puntos_prom"]
todas[f"delta_vs_{referencia}"] = todas["puntos_prom"] - ref["puntos_prom"]
todas["mejora_a_la_referencia"] = (todas[f"delta_vs_{referencia}"] >= 0.5) & (todas["puntos_F3"] >= ref["puntos_F3"])
tabla = todas.sort_values("puntos_prom", ascending=False).reset_index(drop=True)
columnas = ["ronda", "variante", "descripcion", "puntos_prom", "puntos_desv", "puntos_F1", "puntos_F2", "puntos_F3",
            f"delta_vs_{referencia}", "delta_vs_V1", "mejora_a_la_referencia", "auc_prom", "rmse_log_prom",
            "winkler_prom", "cobertura_prom", "archivo"]
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 55)
display(tabla[columnas].round(3))

print("Detalle por corte (ronda 3):")
display(pd.DataFrame(detalle).pivot(index="variante", columns="corte",
                                    values=["auc", "rmse_log", "winkler", "puntos"]).round(3))

if referencia == "E10S":
    print("Aporte de cada grupo espacial (cuánto se pierde al quitarlo):")
    for variante, grupo in [("E10S_sin_poblacion", "población y área"), ("E10S_sin_ubicacion", "ubicación"),
                            ("E10S_sin_vecinos", "tasa de los vecinos")]:
        perdida = ref["puntos_prom"] - r3.set_index("variante").loc[variante, "puntos_prom"]
        print(f"  {grupo:20s}: {perdida:+.2f} puntos " + ("(aporta)" if perdida > 0.2 else "(aporte dudoso)"))

    conjunto_imp = entrenar(FINAL[0], E10S)[0][0]["p"]
    imp = pd.Series(conjunto_imp.booster_.feature_importance("gain"), index=conjunto_imp.feature_name_)
    print("Importancia de variables en el clasificador de E10S (modelo final):")
    display((imp / imp.sum()).sort_values(ascending=False).head(20).round(3).to_frame("importancia"))

tabla.to_csv(f"{CARPETA}/resultados_ronda3.csv", index=False)
candidatas = tabla[tabla["mejora_a_la_referencia"] & (tabla["ronda"] == 3)]
if len(candidatas):
    mejor = candidatas.iloc[0]
    print(f"Mejora a {referencia}: {mejor.variante} ({mejor.descripcion}) · {mejor.puntos_prom:.1f} puntos "
          f"(+{mejor[f'delta_vs_{referencia}']:.1f}) · {mejor.archivo} · ⚠ confirmar cartografía antes de entregar")
else:
    print(f"Ninguna variante mejora ≥ 0.5 puntos a {referencia} sin empeorar F3: mantener {referencia}.")

Reproducibilidad E10S: ronda 2 21.96 vs ronda 3 21.98 ✅
Reproducibilidad E7_lento: ronda 2 21.65 vs ronda 3 21.65 ✅


,ronda,variante,descripcion,puntos_prom,puntos_desv,puntos_F1,puntos_F2,puntos_F3,delta_vs_E10S,delta_vs_V1,mejora_a_la_referencia,auc_prom,rmse_log_prom,winkler_prom,cobertura_prom,archivo
0,3,E10S_arboles_50,Clasificador con 50 árboles,22.135,2.684,21.587,25.050,19.767,0.154,2.071,False,0.703,1.292,3.416,0.920,entregas_ronda3/entrega_AICOLOMBIA_E10S_arboles_50.csv
1,3,ENS_E10S_E7,Promedio fijo de E10S y E7_lento,22.095,2.603,22.165,24.663,19.458,0.115,2.031,False,0.703,1.291,3.409,0.919,entregas_ronda3/entrega_AICOLOMBIA_ENS_E10S_E7.csv
2,3,E10S_k3,Vecinos: 3 más cercanos,22.077,2.704,21.895,24.868,19.468,0.096,2.013,False,0.703,1.292,3.421,0.918,entregas_ronda3/entrega_AICOLOMBIA_E10S_k3.csv
3,3,E10S_lento,Tasa de aprendizaje 0.02 con 250 árboles,22.049,2.763,21.907,24.880,19.361,0.069,1.985,False,0.702,1.291,3.426,0.918,entregas_ronda3/entrega_AICOLOMBIA_E10S_lento.csv
4,2,E9_ensamble_top3,(ronda 2),22.042,2.834,21.975,24.909,19.242,0.061,1.978,False,0.702,1.291,3.411,NaN,zip de la ronda 2
5,3,E10S,Referencia: E10_espacial_simple de la ronda 2,21.981,2.506,21.716,24.608,19.618,0.000,1.917,False,0.701,1.292,3.416,0.920,entregas_ronda3/entrega_AICOLOMBIA_E10S.csv
6,3,E10S_todo_simple,Magnitud y cuantiles también simples (100 árboles),21.970,2.531,21.734,24.611,19.565,-0.011,1.906,False,0.701,1.292,3.418,0.919,entregas_ronda3/entrega_AICOLOMBIA_E10S_todo_simple...
7,3,E10S_3semillas,Promedio de 3 semillas,21.966,2.781,21.749,24.850,19.300,-0.014,1.902,False,0.701,1.291,3.411,0.920,entregas_ronda3/entrega_AICOLOMBIA_E10S_3semillas.csv
8,2,E10_espacial_simple,(ronda 2),21.961,2.813,21.596,24.938,19.348,-0.020,1.897,False,0.701,1.291,3.427,NaN,zip de la ronda 2
9,3,E10S_arboles_150,Clasificador con 150 árboles,21.891,2.583,21.633,24.594,19.447,-0.090,1.827,False,0.701,1.292,3.416,0.920,entregas_ronda3/entrega_AICOLOMBIA_E10S_arboles_150...


Detalle por corte (ronda 3):


auc               rmse_log               winkler                puntos                
corte                       F1     F2     F3       F1     F2     F3      F1     F2     F3      F1      F2      F3
variante                                                                                                         
E10S                     0.697  0.727  0.680    1.352  1.260  1.263   3.501  3.295  3.450  21.716  24.608  19.618
E10S_3semillas           0.697  0.730  0.677    1.351  1.258  1.263   3.490  3.302  3.441  21.749  24.850  19.300
E10S_arboles_150         0.696  0.727  0.678    1.352  1.260  1.263   3.501  3.295  3.450  21.633  24.594  19.447
E10S_arboles_50          0.696  0.732  0.681    1.352  1.260  1.263   3.501  3.295  3.450  21.587  25.050  19.767
E10S_frontera            0.697  0.730  0.676    1.352  1.262  1.266   3.515  3.306  3.440  21.642  24.836  19.173
E10S_hojas_7             0.694  0.728  0.674    1.354  1.256  1.263   3.524  3.347  3.421  21.368  24.668  19.052
E10S_k10                 0.694  0.730  0.672    1.351  1.260  1.265   3.510  3.320  3.444  21.392  24.862  18.831
E10S_k10_ponderado       0.697  0.730  0.676    1.353  1.258  1.266   3.522  3.305  3.447  21.626  24.887  19.148
E10S_k3                  0.699  0.730  0.679    1.351  1.260  1.264   3.510  3.299  3.454  21.895  24.868  19.468
E10S_lento               0.699  0.730  0.678    1.351  1.259  1.263   3.515  3.301  3.462  21.907  24.880  19.361
E10S_sin_poblacion       0.693  0.727  0.672    1.351  1.258  1.266   3.487  3.319  3.443  21.358  24.593  18.783
E10S_sin_ubicacion       0.700  0.721  0.674    1.349  1.259  1.265   3.511  3.329  3.438  21.989  23.912  19.024
E10S_sin_vecinos         0.698  0.727  0.677    1.351  1.259  1.267   3.513  3.315  3.463  21.834  24.520  19.221
E10S_suavizado_espacial  0.691  0.726  0.670    1.352  1.262  1.267   3.510  3.291  3.466  21.121  24.415  18.513
E10S_todo_simple         0.697  0.727  0.680    1.350  1.260  1.267   3.511  3.294  3.449  21.734  24.611  19.565
E10S_vecinos_plus        0.701  0.723  0.675    1.348  1.256  1.264   3.491  3.248  3.440  22.116  24.327  19.134
E7_lento                 0.701  0.723  0.672    1.351  1.260  1.268   3.499  3.262  3.475  22.057  24.199  18.697
ENS_E10S_E7              0.702  0.727  0.679    1.351  1.259  1.265   3.496  3.273  3.457  22.165  24.663  19.458

Aporte de cada grupo espacial (cuánto se pierde al quitarlo):
  población y área    : +0.40 puntos (aporta)
  ubicación           : +0.34 puntos (aporta)
  tasa de los vecinos : +0.12 puntos (aporte dudoso)


c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
c:\Users\ingju\AppData\Local\Programs\Python\Python312\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Importancia de variables en el clasificador de E10S (modelo final):


,importancia
hist_tasa_mes,0.137
hist_esperado_log,0.088
oni_x_tasa,0.077
log_poblacion,0.073
hist_tasa_dep_mes,0.069
hist_tasa_mun,0.063
hist_reciente_12,0.046
departamento,0.042
hist_vecinos_tasa_mes,0.036
ONI,0.028


Ninguna variante mejora ≥ 0.5 puntos a E10S sin empeorar F3: mantener E10S.


## 10. Registro central de experimentos
Agrega una fila por variante a `outputs/reportes/registro_experimentos.csv` (en Drive si está montado; si no, queda una copia dentro del zip).


In [13]:
# Registro central de experimentos: una fila por variante en outputs/reportes/registro_experimentos.csv
import datetime
import hashlib
import json
import sys

NOTEBOOK = "notebook_experimentos3_E10S_AICOLOMBIA"
REGISTRO_DRIVE = "/content/drive/MyDrive/Documentos Juan Felipe/Hackaton_PENUD/outputs/reportes"
COLUMNAS_REGISTRO = ["id_corrida", "fecha_hora", "ronda", "notebook", "entorno", "lightgbm", "huella_datos", "eventos_entrenamiento", "fuente", "variante", "descripcion", "config", "puntos_prom", "puntos_desv", "auc_prom", "rmse_log_prom", "winkler_prom", "cobertura_prom", "factor_q90", "archivo_entrega", "puntos_F1", "puntos_F2", "puntos_F3", "auc_F1", "auc_F2", "auc_F3", "rmse_log_F1", "rmse_log_F2", "rmse_log_F3", "winkler_F1", "winkler_F2", "winkler_F3", "cobertura_F1", "cobertura_F2", "cobertura_F3", "notas"]


def carpeta_registro():
    """Carpeta outputs/reportes del proyecto: en Drive (Colab) o local. None si no se encuentra."""
    if "google.colab" in sys.modules:
        try:
            from google.colab import drive
            if not os.path.isdir("/content/drive/MyDrive"):
                drive.mount("/content/drive")
        except Exception as error:
            print(f"⚠ No se pudo montar Drive: {error}")
        if os.path.isdir(REGISTRO_DRIVE):
            return REGISTRO_DRIVE
    for base in (os.getcwd(), os.path.dirname(os.getcwd())):
        if os.path.isdir(os.path.join(base, "outputs", "reportes")):
            return os.path.join(base, "outputs", "reportes")
    return None


ahora = datetime.datetime.now()
id_corrida = ahora.strftime("%Y%m%d_%H%M%S")
huella = hashlib.md5(f"{len(train)}|{int(train.tiene_evento.sum())}|{train.personas_desplazadas.sum():.0f}"
                     .encode()).hexdigest()[:8]
det = pd.DataFrame(detalle)
filas_registro = []
for r in resumen:
    d = det[det["variante"] == r["variante"]].set_index("corte")
    fila = {"id_corrida": id_corrida, "fecha_hora": f"{ahora:%Y-%m-%d %H:%M}", "ronda": r["ronda"], "notebook": NOTEBOOK,
            "entorno": "colab" if "google.colab" in sys.modules else "local", "lightgbm": lgb.__version__,
            "huella_datos": huella, "eventos_entrenamiento": int(train["tiene_evento"].sum()), "fuente": "ejecución",
            "variante": r["variante"], "descripcion": r["descripcion"],
            "config": json.dumps(VARIANTES.get(r["variante"], {}), default=str, sort_keys=True),
            "archivo_entrega": r["archivo"], "notas": "",
            **{k: r.get(k) for k in ["puntos_prom", "puntos_desv", "auc_prom", "rmse_log_prom", "winkler_prom",
                                     "cobertura_prom", "factor_q90"]}}
    for c in ["F1", "F2", "F3"]:
        for m in ["puntos", "auc", "rmse_log", "winkler", "cobertura"]:
            fila[f"{m}_{c}"] = d.loc[c, m] if c in d.index else np.nan
    filas_registro.append(fila)
nuevo = pd.DataFrame(filas_registro).reindex(columns=COLUMNAS_REGISTRO)
nuevo.to_csv(f"{CARPETA}/registro_experimentos_{id_corrida}.csv", index=False)    # copia que viaja en el zip

carpeta = carpeta_registro()
if carpeta:
    ruta = os.path.join(carpeta, "registro_experimentos.csv")
    previo = pd.read_csv(ruta, dtype={"id_corrida": str}) if os.path.exists(ruta) else None
    total = nuevo if previo is None else pd.concat([previo, nuevo], ignore_index=True)
    total.to_csv(ruta, index=False)
    print(f"Registro central actualizado: +{len(nuevo)} filas → {ruta} ({len(total)} en total)")
else:
    print(f"No se encontró la carpeta del proyecto: el registro quedó en el zip ({CARPETA}/registro_experimentos_{id_corrida}.csv).")
    print("Para sumarlo al registro central: python scripts/consolidar_registro.py <ese archivo>")


Registro central actualizado: +18 filas → g:\My Drive\Documentos Juan Felipe\Hackaton_PENUD\outputs\reportes\registro_experimentos.csv (38 en total)


## 11. Descargar los CSV de la ronda 3


In [14]:
shutil.make_archive("entregas_ronda3", "zip", CARPETA)
try:
    from google.colab import files
    files.download("entregas_ronda3.zip")
except ImportError:
    print("Guardado en", os.path.abspath("entregas_ronda3.zip"))

Guardado en g:\My Drive\Documentos Juan Felipe\Hackaton_PENUD\notebooks\entregas_ronda3.zip
